# Load CSV

In [52]:
import pandas as pd

df = pd.read_csv('csi_data_20260513_095050.csv')
df.head()

,timestamp,type,id,mac,rssi,rate,noise_floor,fft_gain,agc_gain,channel,local_timestamp,sig_len,rx_state,len,first_word,data
0,1.778641e+09,CSI_DATA,61577,08:40:f3:fd:1c:c1,-31,11,-99,-17,29,6,1243475213,83,0,512,0,"[0,0,0,0,0,0,0,0,0,0,0,-1,0,-1,-5,0,-7,1,-12,3..."
1,1.778641e+09,CSI_DATA,61578,08:40:f3:fd:1c:c1,-31,11,-99,-19,29,6,1243475629,83,0,512,0,"[0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,5,3,11,3,17,5..."
2,1.778641e+09,CSI_DATA,61579,08:40:f3:fd:1c:c1,-26,11,-99,16,16,6,1243476299,83,0,512,0,"[0,0,0,0,0,0,0,0,0,0,-6,-9,-6,-9,-12,-6,-18,-3..."
3,1.778641e+09,CSI_DATA,61580,08:40:f3:fd:1c:c1,-31,11,-99,-18,29,6,1243534238,83,0,512,0,"[0,0,0,0,0,0,0,0,0,0,-1,-1,-1,-1,-5,1,-5,5,-9,..."
4,1.778641e+09,CSI_DATA,61581,08:40:f3:fd:1c:c1,-31,11,-99,-21,29,6,1243534566,83,0,512,0,"[0,0,0,0,0,0,0,0,0,0,2,2,2,2,4,-6,4,-10,4,-14,..."


# Parse I/Q → Amplitude & Phase (vectorized)

Kết quả: hai ma trận 2D `amp_matrix` và `phase_matrix` có shape `(n_rows, n_subcarriers)`.
Toàn bộ pipeline sau đều làm việc trên ma trận này thay vì dùng `.apply()` từng row.

In [53]:
import ast
import numpy as np

# ── Subcarrier pilot / DC indices cần loại bỏ ──────────────────────────────
_DROP = {
     64: list(range(0, 6))  + list(range(59, 64)) + [32],
    128: list(range(0, 6))  + list(range(122, 128)) + [63, 64, 65],
    256: list(range(0, 11)) + list(range(245, 256)) + [127, 128, 129],
}

def _parse_row(value, n_sub):
    """Parse chuỗi/list I/Q → (i, q) sau khi loại pilot/DC."""
    if isinstance(value, str):
        value = ast.literal_eval(value)
    arr = np.asarray(value, dtype=float)
    i, q = arr[0::2], arr[1::2]

    drop = _DROP.get(n_sub)
    if drop:
        mask = np.ones(n_sub, dtype=bool)
        mask[drop] = False
        i, q = i[mask], q[mask]
    return i, q


# ── Vectorized: build 2D matrices trực tiếp ────────────────────────────────
amps, phases = [], []

for _, row in df.iterrows():
    n_sub = int(row["len"]) // 2
    i, q = _parse_row(row["data"], n_sub)
    amps.append(np.hypot(i, q))
    phases.append(np.arctan2(q, i))

# shape: (n_rows, n_subcarriers)
amp_matrix   = np.stack(amps)    
phase_matrix = np.stack(phases)  

print(f"amp_matrix  : {amp_matrix.shape}")
print(f"phase_matrix: {phase_matrix.shape}")

amp_matrix  : (4000, 231)
phase_matrix: (4000, 231)


In [54]:
# Test detect by variance
import pandas as pd
import numpy as np
import plotly.graph_objects as go

baseline_csv = 'Router/khong nguoi/csi_data_20260313_120718.csv'
test_csv     = 'Router/khong nguoi/csi_data_20260313_154847.csv'


def build_amp_matrix(csv_path):
    df_local = pd.read_csv(csv_path)
    amps_local = []
    for _, row in df_local.iterrows():
        n_sub = int(row['len']) // 2
        i, q = _parse_row(row['data'], n_sub)
        amps_local.append(np.hypot(i, q))
    return np.stack(amps_local)

amp_person = build_amp_matrix(test_csv)
amp_empty = build_amp_matrix(baseline_csv)

# Tính variance cho mỗi dataset
var_empty = amp_empty.var(axis=1).mean()
var_person = amp_person.var(axis=1).mean()

# Threshold = trung bình hai giá trị
threshold = (var_empty + var_person) * 1.2 / 2

print(f'Baseline file : {baseline_csv}')
print(f'var_empty     : {var_empty:.6f}')
print(f'var_person    : {var_person:.6f}')
print(f'threshold     : {threshold:.6f}')
print(f'person > thr? : {var_person > threshold}')
print(f'empty  < thr? : {var_empty < threshold}')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['empty', 'person'],
    y=[var_empty, var_person],
    marker_color=['#4C78A8', '#F58518'],
    name='variance',
))
fig.add_hline(
    y=threshold,
    line_dash='dash',
    line_color='red',
    annotation_text=f'threshold={threshold:.6f}',
    annotation_position='top right',
)
fig.update_layout(
    title='Variance-based Detect Test',
    yaxis_title='Mean row variance',
    xaxis_title='Dataset',
    height=420,
)
fig.show()


Baseline file : Router/khong nguoi/csi_data_20260313_120718.csv
var_empty     : 1724.345040
var_person    : 3308.502961
threshold     : 3019.708801
person > thr? : True
empty  < thr? : True


# Phase Unwrap & Sanitize (theo chiều subcarrier / spatial)

Unwrap loại bỏ jump 2π **trên chiều subcarrier** (mỗi row = một snapshot không gian).  
Sau đó trừ trend tuyến tính để loại bỏ phase offset tổng thể.

In [55]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

person_csv = 'csi_data_20260513_094619.csv'
baseline_csv = 'router no person/csi_data_20260313_121001.csv'

def build_amp_matrix(csv_path):
    df_local = pd.read_csv(csv_path)
    amps_local = []
    for _, row in df_local.iterrows():
        n_sub = int(row['len']) // 2
        i, q = _parse_row(row['data'], n_sub)
        amps_local.append(np.hypot(i, q))
    return np.stack(amps_local)

amp_person = amp_matrix
amp_empty = build_amp_matrix(baseline_csv)

# Tính variance cho mỗi dataset
var_empty = amp_empty.var(axis=1).mean()
var_person = amp_person.var(axis=1).mean()

# Threshold = trung bình hai giá trị
threshold = (var_empty + var_person) / 2

print(f'Person file   : {person_csv}')
print(f'Baseline file  : {baseline_csv}')
print(f'var_empty      : {var_empty:.6f}')
print(f'var_person     : {var_person:.6f}')
print(f'threshold      : {threshold:.6f}')
print(f'detect person? : {var_person > threshold}')
print(f'detect empty?  : {var_empty < threshold}')

fig = go.Figure()
fig.add_trace(go.Bar(
    x=['empty', 'person'],
    y=[var_empty, var_person],
    marker_color=['#4C78A8', '#F58518'],
    name='variance',
))
fig.add_hline(
    y=threshold,
    line_dash='dash',
    line_color='red',
    annotation_text=f'threshold={threshold:.6f}',
    annotation_position='top right',
)
fig.update_layout(
    title='Variance-based Detect Test',
    yaxis_title='Mean row variance',
    xaxis_title='Dataset',
    height=420,
)
fig.show()
import numpy as np

# ── Bước 1: Spatial unwrap + sanitize (per row, như cũ) ──────────────────
def _unwrap_sanitize_row(row_phase):
    """Loại 2π jump và timing offset theo chiều subcarrier."""
    u = np.unwrap(row_phase)
    k = np.arange(u.size)
    a, b = np.polyfit(k, u, 1)
    return u - (a * k + b)

phase_spatial = np.apply_along_axis(_unwrap_sanitize_row, axis=1, arr=phase_matrix)

# ── Bước 2: Temporal unwrap (per subcarrier, theo chiều thời gian) ────────
# np.unwrap(axis=0) → unwrap từng cột (subcarrier) qua thời gian
phase_unwrapped = np.unwrap(phase_spatial, axis=0)

Person file   : csi_data_20260513_094619.csv
Baseline file  : router no person/csi_data_20260313_121001.csv
var_empty      : 1728.170633
var_person     : 2622.742916
threshold      : 2175.456774
detect person? : True
detect empty?  : True


# Hampel Filter (theo chiều thời gian / temporal)

**Bug v2**: filter được apply từng row → lọc qua chiều subcarrier (sai).  
**Fix**: apply theo `axis=0` → mỗi cột = chuỗi thời gian của một subcarrier.

**Bug v2**: `from hampel_filter import hampel_filter as hf` → import sai module.  
**Fix**: `from hampel_filter import hampel`.

In [56]:
# FIX: import đúng
from hampel_filter import hampel
import numpy as np

def _hampel_1d(arr, window_size=20, n_sigma=3.0):
    """Hampel filter cho một chuỗi 1D, thay outlier bằng median cửa sổ (loại outlier ra)."""
    arr = np.asarray(arr, dtype=float)
    if arr.size == 0:
        return arr

    outlier_indices = hampel(
        arr,
        window_size=window_size,
        n=n_sigma,
        parallel=False,   # parallel=True dễ gây overhead với arr nhỏ
        return_indices=True,
    )

    if isinstance(outlier_indices, tuple):
        outlier_indices = outlier_indices[0]
    outlier_indices = np.asarray(outlier_indices, dtype=int)

    if outlier_indices.size == 0:
        return arr.copy()

    filtered = arr.copy()
    for idx in outlier_indices:
        start = max(0, idx - window_size)
        end   = min(arr.size, idx + window_size + 1)
        # FIX: loại chính điểm outlier ra trước khi tính median
        window = np.concatenate([arr[start:idx], arr[idx + 1:end]])
        filtered[idx] = np.median(window)
    return filtered


# FIX: apply_along_axis(axis=0) → theo chiều thời gian (đúng cho temporal filter)
amp_hampel   = np.apply_along_axis(_hampel_1d, axis=0, arr=amp_matrix)
phase_hampel = np.apply_along_axis(_hampel_1d, axis=0, arr=phase_unwrapped)

print(f"amp_hampel  : {amp_hampel.shape}")
print(f"phase_hampel: {phase_hampel.shape}")

amp_hampel  : (4000, 231)
phase_hampel: (4000, 231)


# Savitzky-Golay Filter (theo chiều thời gian / temporal)

In [57]:
from scipy.signal import savgol_filter
import numpy as np

def _savgol_1d(arr, window_length=31, polyorder=3):
    arr = np.asarray(arr, dtype=float)
    if arr.size < 3:
        return arr
    wl = min(window_length, arr.size)
    if wl % 2 == 0:
        wl -= 1
    if wl < 3:
        return arr
    po = min(polyorder, wl - 1)
    return savgol_filter(arr, window_length=wl, polyorder=po)

# apply_along_axis(axis=0) → theo chiều thời gian
amp_sg   = np.apply_along_axis(_savgol_1d, axis=0, arr=amp_hampel)
phase_sg = np.apply_along_axis(_savgol_1d, axis=0, arr=phase_hampel)

print(f"amp_sg  : {amp_sg.shape}")
print(f"phase_sg: {phase_sg.shape}")

amp_sg  : (4000, 231)
phase_sg: (4000, 231)


In [58]:
# from scipy.signal import butter, sosfiltfilt, detrend
# import numpy as np

# # High-pass very low cutoff to remove DC + ultra-slow drift
# HP_CUTOFF = 0.05  # Hz
# HP_ORDER = 2

# # _hp_sos = butter(HP_ORDER, HP_CUTOFF, btype="highpass", fs=FS, output="sos")
# HP_MIN_LEN = HP_ORDER * 3 + 1

# def _highpass_1d(arr):
#     arr = np.asarray(arr, dtype=float)
#     if arr.size < HP_MIN_LEN:
#         return arr
#     arr_detrended = detrend(arr, type="constant")
#     return sosfiltfilt(_hp_sos, arr_detrended)

# # Apply along time axis
# amp_hp   = np.apply_along_axis(_highpass_1d, axis=0, arr=amp_sg)
# phase_hp = np.apply_along_axis(_highpass_1d, axis=0, arr=phase_sg)

# print(f"amp_hp  : {amp_hp.shape}")
# print(f"phase_hp: {phase_hp.shape}")

# Butterworth Bandpass Filter (theo chiều thời gian / temporal)

**Bug v2**: `butter(order, [low, high], fs=fs)` trong khi `low = lowcut/nyq` → double normalization!  
**Fix**: bỏ tính tay `low/high`, truyền thẳng Hz vào `butter(..., fs=fs)`.

In [59]:
from scipy.signal import butter, sosfiltfilt, detrend
import numpy as np

FS      = 100.0
LOWCUT  = 0.1
HIGHCUT = 0.5
ORDER   = 4

# FIX 1: dùng SOS thay vì ba — ổn định số học hơn
_sos = butter(ORDER, [LOWCUT, HIGHCUT], btype='band', fs=FS, output='sos')
MIN_LEN = ORDER * 3 + 1

def _butter_1d(arr):
    arr = np.asarray(arr, dtype=float)
    if arr.size < MIN_LEN:
        return arr
    
    # FIX 2: detrend trước khi lọc — loại bỏ DC offset và linear trend
    arr_detrended = detrend(arr, type='linear')
    
    return sosfiltfilt(_sos, arr_detrended)

amp_bw   = np.apply_along_axis(_butter_1d, axis=0, arr=amp_sg)
phase_bw = np.apply_along_axis(_butter_1d, axis=0, arr=phase_sg)

# Highpass

# Visualize Pipeline

In [60]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sub_idx   = 10
row_start = 0
row_end   = 4000

end = min(row_end, amp_matrix.shape[0])
x   = np.arange(row_start, end)

def _col(matrix):
    """Lấy cột sub_idx từ ma trận 2D."""
    if matrix is None or matrix.shape[1] <= sub_idx:
        return None
    return matrix[row_start:end, sub_idx]

amp_stages = [
    ("raw",      amp_matrix),
    ("hampel",   amp_hampel),
    ("sg",       amp_sg),
    ("butter",   amp_bw),
]

phase_stages = [
    ("raw",             phase_matrix),
    ("unwrap_sanitize", phase_unwrapped),
    ("hampel",          phase_hampel),
    ("sg",              phase_sg),
    ("butter",          phase_bw),
]

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Amplitude pipeline", "Phase pipeline"),
)

for label, mat in amp_stages:
    y = _col(mat)
    if y is None:
        continue
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=f"amp_{label}"), row=1, col=1)

for label, mat in phase_stages:
    y = _col(mat)
    if y is None:
        continue
    fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name=f"phase_{label}"), row=2, col=1)

fig.update_layout(
    height=700,
    title=f"Subcarrier {sub_idx} — rows {row_start}–{end - 1}",
    xaxis_title="row",
    xaxis2_title="row",
    legend_title="stage",
)
fig.show()

# PCA

In [61]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import numpy as np

def _fit_pca(matrix, name, n_components=5):
    X_scaled = StandardScaler().fit_transform(matrix)
    pca = PCA(n_components=n_components, svd_solver="randomized", random_state=42)
    X_pca = pca.fit_transform(X_scaled)

    print(f"\n── {name} ──")
    for i, (e, ec) in enumerate(zip(pca.explained_variance_ratio_,
                                     np.cumsum(pca.explained_variance_ratio_))):
        print(f"  PC{i+1}: {e*100:5.2f}%  cumulative: {ec*100:6.2f}%")
    return X_pca, pca

amp_pca,   pca_amp   = _fit_pca(amp_bw,   "Amplitude PCA")
phase_pca, pca_phase = _fit_pca(phase_bw, "Phase PCA")

# amp_pca.shape   → (n_rows, 5)
# phase_pca.shape → (n_rows, 5)


── Amplitude PCA ──
  PC1: 38.02%  cumulative:  38.02%
  PC2: 27.75%  cumulative:  65.78%
  PC3: 14.55%  cumulative:  80.33%
  PC4:  4.54%  cumulative:  84.87%
  PC5:  3.34%  cumulative:  88.21%

── Phase PCA ──
  PC1: 18.12%  cumulative:  18.12%
  PC2: 11.98%  cumulative:  30.09%
  PC3: 10.26%  cumulative:  40.35%
  PC4:  8.52%  cumulative:  48.88%
  PC5:  6.12%  cumulative:  55.00%


# Visualize PCA

In [62]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

x = np.arange(amp_pca.shape[0])

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Amplitude PCA", "Phase PCA"),
)

for i in range(amp_pca.shape[1]):
    e = pca_amp.explained_variance_ratio_[i]
    fig.add_trace(
        go.Scatter(x=x, y=amp_pca[:, i], mode="lines", name=f"amp PC{i+1} ({e*100:.1f}%)"),
        row=1, col=1,
    )

for i in range(phase_pca.shape[1]):
    e = pca_phase.explained_variance_ratio_[i]
    fig.add_trace(
        go.Scatter(x=x, y=phase_pca[:, i], mode="lines", name=f"phase PC{i+1} ({e*100:.1f}%)"),
        row=2, col=1,
    )

fig.update_layout(
    height=700,
    title="PCA Components theo thời gian",
    xaxis2_title="row",
    legend_title="component",
)
fig.show()

In [63]:
# from numpy.fft import rfft, rfftfreq

# signal = amp_pca[:, 1]
# freqs  = rfftfreq(len(signal), d=1/FS)
# power  = np.abs(rfft(signal)) ** 2

# # Chỉ xét dải nhịp thở
# mask = (freqs >= 0.1) & (freqs <= 0.5)
# peak_freq = freqs[mask][np.argmax(power[mask])]

# print(f"Tần số dominant: {peak_freq:.3f} Hz")
# print(f"Nhịp thở ước tính: {peak_freq * 60:.1f} nhịp/phút")

# Doppler Shift

In [64]:
from scipy.signal import stft
import numpy as np
import plotly.graph_objects as go

# ── Config ────────────────────────────────────────────────────────────────
FREQ_WIFI  = 2.4e9        # Hz — đổi thành 5e9 nếu dùng 5GHz
LAMBDA     = 3e8 / FREQ_WIFI   # wavelength ~ 0.125m (2.4GHz)

# ── 1. Tính Doppler velocity từ phase derivative ──────────────────────────
# v = (Δφ / Δt) * λ / (4π)
# Dùng PC1 của phase (signal tổng hợp nhất)
phase_signal = phase_pca[:, 1]

dphi = np.diff(phase_signal)          # Δφ giữa các sample
dt   = 1.0 / FS

velocity = (dphi / dt) * LAMBDA / (4 * np.pi)   # m/s

print(f"Velocity range: [{velocity.min():.4f}, {velocity.max():.4f}] m/s")
print(f"Max speed: {np.abs(velocity).max()*100:.2f} cm/s")

# ── 2. Micro-Doppler spectrogram (STFT) ───────────────────────────────────
# STFT cho thấy velocity thay đổi theo thời gian → pattern nhịp thở
nperseg = int(FS * 4)    # cửa sổ 4 giây
noverlap = int(nperseg * 0.9)

f, t, Zxx = stft(phase_signal, fs=FS, nperseg=nperseg, noverlap=noverlap, detrend='constant')

# Chỉ lấy dải tần số nhịp thở
mask = (f >= -2.0) & (f <= 2.0)   # Hz — tương đương ±25 cm/s ở 2.4GHz
f_breath = f[mask]
Z_breath = np.abs(Zxx[mask, :])

# Convert tần số → velocity
v_axis = f_breath * LAMBDA / 2    # m/s (Doppler formula: v = f_d * λ / 2)

# ── 3. Plot ───────────────────────────────────────────────────────────────
fig = go.Figure(go.Heatmap(
    x=t,
    y=v_axis * 100,
    z=20 * np.log10(Z_breath + 1e-10),
    colorscale="Jet",
    colorbar=dict(title="dB"),
    # zmin=-10,
    # zmax=-10,
))
fig.update_layout(
    title="Micro-Doppler Spectrogram — Phase PC1",
    xaxis_title="Thời gian (s)",
    yaxis_title="Velocity (cm/s)",
    height=400,
)
fig.show()

# ── 4. Ước tính nhịp thở từ dominant velocity theo thời gian ─────────────
dominant_v = v_axis[np.argmax(Z_breath, axis=0)]   # velocity mạnh nhất tại mỗi frame

# Tổng energy theo velocity tại mỗi frame → breathing envelope
breath_envelope = np.sum(Z_breath, axis=0)  # shape (n_frames,)

# FFT để tìm nhịp thở
from numpy.fft import rfft, rfftfreq
freqs = rfftfreq(len(breath_envelope), d=t[1]-t[0])
power = np.abs(rfft(breath_envelope)) ** 2

mask2 = (freqs >= 0.1) & (freqs <= 0.5)
peak_freq = freqs[mask2][np.argmax(power[mask2])]
print(f"Nhịp thở: {peak_freq * 60:.1f} nhịp/phút")

Velocity range: [-0.1938, 0.1896] m/s
Max speed: 19.38 cm/s


Nhịp thở: 8.9 nhịp/phút
